In [0]:
from LDCDataAccessLayerPy import KeyVaultManager, SharePointManager, SqlManager, databricks_init
from datetime import datetime, timedelta
from LDCDataAccessLayerPy import databricks_init, DataLakeManagerGen2
from io import BytesIO
import LDCDataAccessLayerPy
#Initiate the secret to access KeyVault secrets
databricks_init(dbutils, 'GO')
sp_mgr = SharePointManager()
sql_mgr = SqlManager()

import logging
logger = spark._jvm.org.apache.log4j
logging.getLogger("py4j").setLevel(logging.ERROR)

import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.colors import ListedColormap
from sklearn.cluster import KMeans
from openpyxl import load_workbook

from datetime import datetime, timedelta
import re
from dateutil.relativedelta import relativedelta
from LDCDataAccessLayerPy import PriceManager, graph
from LDCDataAccessLayerPy import ZemaManager
import math
import plotly.express as px
import plotly.graph_objects as go
zema = ZemaManager()

url = "https://ldcom365.sharepoint.com"


In [0]:
start_date = datetime(2017,8, 1)

end_date = datetime(2030, 1, 1)

In [0]:
corn_matba= zema.get_curve(curve="P-FUTURE-MATBA-INPUT-CORN-USD-MT", period=f"{start_date}::{end_date}")
corn_matba=corn_matba[corn_matba['observation']=='Settle']

corn_matba=corn_matba[['date','value','contract_year','contract_month']]
corn_matba['day']=corn_matba['date'].dt.day
corn_matba['month']=corn_matba['date'].dt.month
corn_matba['year']=corn_matba['date'].dt.year
corn_matba['virtual_date'] = pd.to_datetime(
    {'year': 2000, 'month': corn_matba['month'], 'day': corn_matba['day']},
    errors='coerce'  # This will convert invalid dates (e.g. Feb 30) to NaT
)
# Drop rows with invalid virtual dates
corn_matba = corn_matba.dropna(subset=['virtual_date'])

# You can now sort or use this date for seasonal charts
corn_matba = corn_matba.sort_values(by='virtual_date')
corn_matba=corn_matba[['date','value','contract_year','contract_month','virtual_date']]
corn_matba['contract_date'] = (
    pd.to_datetime(dict(year=corn_matba['contract_year'],
                        month=corn_matba['contract_month'],
                        day=1))  # temporary first day of month
    + pd.offsets.MonthEnd(0)   # shift to last day of that month
)

sp_mgr.save_pd_to_excel('/sites/grainsargprojects/models/zema_prices/CORN/corn_matba.xlsx',corn_matba,index=False)

In [0]:
corn_matba_APR=corn_matba[corn_matba['contract_month']==4]

corn_matba_APR['Season'] = 'Apr'+ corn_matba_APR['contract_year'].astype(str)
  
def adjust_virtual_date_corn_apr(row):
    if row['virtual_date'].month in [1, 2,3,4]:
        # Subtract 1 year from the year if month is November or December
        return row['virtual_date'] + pd.DateOffset(years=1)
    else:
        # Keep the date as is if the month is not November or December
        return row['virtual_date'].replace(year=row['virtual_date'].year)
      
corn_matba_APR['virtual_date'] = corn_matba_APR.apply(adjust_virtual_date_corn_apr, axis=1)
corn_matba_APR = corn_matba_APR[corn_matba_APR['value'] != 0]

corn_matba_APR = corn_matba_APR[
    (corn_matba_APR['date'].dt.month.isin([1, 2, 3, 4]) & (corn_matba_APR['date'].dt.year == corn_matba_APR['contract_year'])) |
    (~corn_matba_APR['date'].dt.month.isin([1, 2, 3, 4]) & (corn_matba_APR['date'].dt.year +1 == corn_matba_APR['contract_year']))
]
corn_matba_APR = corn_matba_APR.sort_values(by='date')


apr_matba = px.line(
    corn_matba_APR,
    x='virtual_date',
    y='value',
    color='Season',
    labels={
        'virtual_date': '',
        'value': 'Spot Price',
        'season': 'Year'
    },
    title='Apr MATBA contract',
)

apr_matba.update_traces(hovertemplate='%{x|%d-%b}<br>%{y:.2f}<extra>%{fullData.name}</extra>')
apr_matba.update_layout(
    xaxis=dict(
        tickformat='%b',  # Month format like Jan, Feb...
        dtick="M1",
        hoverformat='%d-%b'
    ),
    yaxis_title='Price',
    template='plotly_white',
    height=700,
    width=1800,
    hovermode='x unified'
    
)

# Customize specific seasons
for trace in apr_matba.data:
    if trace.name == 'Apr2026':
        trace.line.width = 4
        trace.line.color = 'red'
        trace.line.dash = 'dash'
    elif trace.name == 'Apr2025':
        trace.line.width = 4
        trace.line.color = 'black'
        trace.line.dash = 'solid'

apr_matba.show()


apr_matba_html = apr_matba.to_html(include_plotlyjs='cdn', full_html=True)

# sp_mgr.save_pd_to_excel('/sites/grainsargprojects/models/zema_prices/CORN/corn_matba_dec.xlsx',corn_matba_DEC,index=False)


In [0]:
soy_matba= zema.get_curve(curve="P-FUTURE-MATBA-INPUT-SOYBEAN-USD-MT", period=f"{start_date}::{end_date}")
soy_matba=soy_matba[soy_matba['observation']=='Settle']

soy_matba=soy_matba[['date','value','contract_year','contract_month']]
soy_matba['day']=soy_matba['date'].dt.day
soy_matba['month']=soy_matba['date'].dt.month
soy_matba['year']=soy_matba['date'].dt.year
soy_matba['virtual_date'] = pd.to_datetime(
    {'year': 2000, 'month': soy_matba['month'], 'day': soy_matba['day']},
    errors='coerce'  # This will convert invalid dates (e.g. Feb 30) to NaT
)
# Drop rows with invalid virtual dates
soy_matba = soy_matba.dropna(subset=['virtual_date'])

# You can now sort or use this date for seasonal charts
soy_matba = soy_matba.sort_values(by='virtual_date')
soy_matba=soy_matba[['date','value','contract_year','contract_month','virtual_date']]
soy_matba['contract_date'] = (
    pd.to_datetime(dict(year=soy_matba['contract_year'],
                        month=soy_matba['contract_month'],
                        day=1))  # temporary first day of month
    + pd.offsets.MonthEnd(0)   # shift to last day of that month
)

soy_matba=soy_matba.sort_values(by='date')
soy_matba

sp_mgr.save_pd_to_excel('/sites/grainsargprojects/models/zema_prices/CORN/soy_matba.xlsx',soy_matba,index=False)

In [0]:
may = soy_matba[soy_matba["contract_month"] == 5].copy()
apr=corn_matba[corn_matba["contract_month"] == 4].copy()

apr_match = (apr.groupby("date")
                .apply(lambda x: x.loc[x["contract_date"] >= x.name].sort_values("contract_date").head(1))
                .reset_index(drop=True))

may_match = (may.groupby("date")
                .apply(lambda x: x.loc[x["contract_date"] >= x.name].sort_values("contract_date").head(1))
                .reset_index(drop=True))

may_match.drop(["contract_year", "contract_month", "contract_date","virtual_date"], axis=1, inplace=True)
may_match.rename(columns={"value": "SB"}, inplace=True)


apr_match.drop(["contract_year", "contract_month", "contract_date","virtual_date"], axis=1, inplace=True)
apr_match.rename(columns={"value": "CORN"}, inplace=True)


# Merge Apr + Jul
ratio_df_apr_may = pd.merge(may_match,apr_match, on="date")

ratio_df_apr_may = ratio_df_apr_may[ratio_df_apr_may['CORN'] != 0]

# Compute spread
ratio_df_apr_may["ratio"] = ratio_df_apr_may["SB"]/ratio_df_apr_may["CORN"]
ratio_df_apr_may['day']=ratio_df_apr_may['date'].dt.day
ratio_df_apr_may['month']=ratio_df_apr_may['date'].dt.month
ratio_df_apr_may['year']=ratio_df_apr_may['date'].dt.year
ratio_df_apr_may['virtual_date'] = pd.to_datetime(
    {'year': 2000, 'month': ratio_df_apr_may['month'], 'day': ratio_df_apr_may['day']},
    errors='coerce'  # This will convert invalid dates (e.g. Feb 30) to NaT
)

ratio_df_apr_may = ratio_df_apr_may[
    ((ratio_df_apr_may['month'] >= 6) & (ratio_df_apr_may['month'] <= 12)) |
    (ratio_df_apr_may['month'] <= 4)
]

# Adjust virtual_date: subtract 1 year for Oct–Dec
ratio_df_apr_may.loc[ratio_df_apr_may['month'] >= 6, 'virtual_date'] = (
    ratio_df_apr_may.loc[ratio_df_apr_may['month'] >= 6, 'virtual_date']
    - pd.DateOffset(years=1)
)


ratio_df_apr_may['season'] = np.where(ratio_df_apr_may['month'] >= 6, ratio_df_apr_may['year'] + 1,ratio_df_apr_may['year'])

# Drop rows with invalid virtual dates
ratio_df_apr_may = ratio_df_apr_may.dropna(subset=['virtual_date'])
ratio_df_apr_may

In [0]:
import plotly.express as px

# Create the base plot
ratio_apr_july = px.line(
    ratio_df_apr_may,
    x="virtual_date",
    y="ratio",
    color="season",
    labels={
        "virtual_date": "",
        "ratio": "MATBA Ratios SB MAY/CORN APR ",
        "season": "season"
    },
    title="SB MAY/CORN APR",
)

# Compute average across seasons
avg_df = ratio_df_apr_may.groupby("virtual_date", as_index=False)["ratio"].mean()
ratio_apr_july.add_scatter(
    x=avg_df["virtual_date"],
    y=avg_df["ratio"],
    mode="lines",
    name="Average",
    line=dict(color="gray", width=3, dash="dash")  # dashed gray line
)

# Identify last two seasons
seasons_sorted = sorted(ratio_df_apr_may["season"].unique())
last_season = seasons_sorted[-1]
prev_season = seasons_sorted[-2]

# Update traces: highlight last and previous seasons
for trace in ratio_apr_july.data:
    if trace.name == str(last_season):
        trace.line.color = "red"
        trace.line.dash = "dash"
        trace.line.width = 5
    elif trace.name == str(prev_season):
        trace.line.color = "black"
        trace.line.dash = "solid"
        trace.line.width = 3

# Layout and hover
ratio_apr_july.update_traces(
    hovertemplate="%{x|%d-%b}<br>%{y:.2f}<extra>%{fullData.name}</extra>"
)
ratio_apr_july.update_layout(
    xaxis=dict(
        tickformat="%b",
        dtick="M1",
        hoverformat="%d-%b"
    ),
    yaxis_title="Price",
    template="plotly_white",
    height=700,
    width=1800,
    hovermode="x unified"
)

ratio_apr_july.show()

ratio_apr_july_html = ratio_apr_july.to_html(include_plotlyjs="cdn", full_html=True)


In [0]:
import plotly.graph_objects as go
import plotly.express as px

seasons = ratio_df_apr_may["season"].unique()
line_styles = {
    "ratio": dict(dash="solid", width=2),
    "SB": dict(dash="dot", width=2),
    "CORN": dict(dash="dash", width=2)
}
color_map = {season: px.colors.qualitative.Plotly[i % len(px.colors.qualitative.Plotly)]
             for i, season in enumerate(seasons)}

fig = go.Figure()

# Add all traces
for season in seasons:
    df_season = ratio_df_apr_may[ratio_df_apr_may["season"] == season]
    # ratio
    fig.add_trace(go.Scatter(
        x=df_season["virtual_date"],
        y=df_season["ratio"],
        mode="lines",
        name=f"{season} - Ratio",
        line=dict(color=color_map[season], **line_styles["ratio"]),
        yaxis="y1",
        visible=True
    ))
    # SB
    fig.add_trace(go.Scatter(
        x=df_season["virtual_date"],
        y=df_season["SB"],
        mode="lines",
        name=f"{season} - SB",
        line=dict(color=color_map[season], **line_styles["SB"]),
        yaxis="y2",
        visible=True
    ))
    # CORN
    fig.add_trace(go.Scatter(
        x=df_season["virtual_date"],
        y=df_season["CORN"],
        mode="lines",
        name=f"{season} - CORN",
        line=dict(color=color_map[season], **line_styles["CORN"]),
        yaxis="y2",
        visible=True
    ))

# Create buttons to toggle each season
buttons = []
for i, season in enumerate(seasons):
    # Determine which traces belong to this season
    visible = [False] * len(fig.data)
    for j, trace in enumerate(fig.data):
        if trace.name.startswith(str(season)):
            visible[j] = True
    buttons.append(dict(
        label=str(season),
        method="update",
        args=[{"visible": visible},
              {"title": f"SB MAY / CORN APR (Showing {season})"}]
    ))
buttons.append(dict(
    label="Show All",
    method="update",
    args=[{"visible": [True]*len(fig.data)},  # all traces visible
          {"title": "SB MAY / CORN APR (All Seasons)"}]
))

# Layout
fig.update_layout(
    updatemenus=[dict(
        type="dropdown",
        active=0,
        buttons=buttons,
        x=0.95,
        y=1.15,
        xanchor="left",
        yanchor="top"
    )],
    title=f"SB MAY / CORN APR",
    template="plotly_white",
    height=700,
    width=1800,
    hovermode="x unified",
    xaxis=dict(tickformat="%b", dtick="M1", hoverformat="%d-%b"),
    yaxis=dict(title="Ratio", side="left"),
    yaxis2=dict(title="Prices (SB & CORN)", overlaying="y", side="right")
)

fig.show()

fig_html = fig.to_html(include_plotlyjs="cdn", full_html=True)


# WHEAT

In [0]:
wheat_matba= zema.get_curve(curve="P-FUTURE-MATBA-INPUT-WHEAT-ROS-USD-MT", period=f"{start_date}::{end_date}")
wheat_matba=wheat_matba[wheat_matba['observation']=='Settle']

wheat_matba=wheat_matba[['date','value','contract_year','contract_month']]
wheat_matba['day']=wheat_matba['date'].dt.day
wheat_matba['month']=wheat_matba['date'].dt.month
wheat_matba['year']=wheat_matba['date'].dt.year
wheat_matba['virtual_date'] = pd.to_datetime(
    {'year': 2000, 'month': wheat_matba['month'], 'day': wheat_matba['day']},
    errors='coerce'  # This will convert invalid dates (e.g. Feb 30) to NaT
)
# Drop rows with invalid virtual dates
wheat_matba = wheat_matba.dropna(subset=['virtual_date'])

# You can now sort or use this date for seasonal charts
wheat_matba = wheat_matba.sort_values(by='virtual_date')
wheat_matba=wheat_matba[['date','value','contract_year','contract_month','virtual_date']]
wheat_matba['contract_date'] = (
    pd.to_datetime(dict(year=wheat_matba['contract_year'],
                        month=wheat_matba['contract_month'],
                        day=1))  # temporary first day of month
    + pd.offsets.MonthEnd(0)   # shift to last day of that month
)
wheat_matba

sp_mgr.save_pd_to_excel('/sites/grainsargprojects/models/zema_prices/CORN/wheat_matba.xlsx',wheat_matba,index=False)

## DEC WHEAT / DEC CORN

In [0]:
dec_w = wheat_matba[wheat_matba["contract_month"] == 12].copy()
dec_c= corn_matba[corn_matba["contract_month"] == 12].copy()

dec_w_match = (dec_w.groupby("date")
                .apply(lambda x: x.loc[x["contract_date"] >= x.name].sort_values("contract_date").head(1))
                .reset_index(drop=True))

dec_c_match = (dec_c.groupby("date")
                .apply(lambda x: x.loc[x["contract_date"] >= x.name].sort_values("contract_date").head(1))
                .reset_index(drop=True))

dec_c_match.drop(["contract_year", "contract_month", "contract_date","virtual_date"], axis=1, inplace=True)
dec_c_match.rename(columns={"value": "CORN"}, inplace=True)


dec_w_match.drop(["contract_year", "contract_month", "contract_date","virtual_date"], axis=1, inplace=True)
dec_w_match.rename(columns={"value": "WHEAT"}, inplace=True)


# Merge dec_w + Jul
ratio_w_c = pd.merge(dec_c_match,dec_w_match, on="date")

ratio_w_c = ratio_w_c[ratio_w_c['CORN'] != 0]
ratio_w_c = ratio_w_c[ratio_w_c['WHEAT'] != 0]

# Compute spread
ratio_w_c["ratio"] = ratio_w_c["WHEAT"]/ratio_w_c["CORN"]
ratio_w_c['day']=ratio_w_c['date'].dt.day
ratio_w_c['month']=ratio_w_c['date'].dt.month
ratio_w_c['year']=ratio_w_c['date'].dt.year
ratio_w_c['virtual_date'] = pd.to_datetime(
    {'year': 2000, 'month': ratio_w_c['month'], 'day': ratio_w_c['day']},
    errors='coerce'  # This will convert invalid dates (e.g. Feb 30) to NaT
)


ratio_w_c['season'] = ratio_w_c['year']

# Drop rows with invalid virtual dates
ratio_w_c = ratio_w_c.dropna(subset=['virtual_date'])
ratio_w_c

In [0]:
# Create the base plot
fig_ratio_w_c = px.line(
    ratio_w_c,
    x="virtual_date",
    y="ratio",
    color="season",
    labels={
        "virtual_date": "",
        "ratio": "MATBA Ratios DEC WHEAT/DEC CORN ",
        "season": "season"
    },
    title="DEC WHEAT/DEC CORN",
)

# Compute average across seasons
avg_df = ratio_w_c.groupby("virtual_date", as_index=False)["ratio"].mean()
fig_ratio_w_c.add_scatter(
    x=avg_df["virtual_date"],
    y=avg_df["ratio"],
    mode="lines",
    name="Average",
    line=dict(color="gray", width=3, dash="dash")  # dashed gray line
)

# Identify last two seasons
seasons_sorted = sorted(ratio_w_c["season"].unique())
last_season = seasons_sorted[-1]
prev_season = seasons_sorted[-2]

# Update traces: highlight last and previous seasons
for trace in fig_ratio_w_c.data:
    if trace.name == str(last_season):
        trace.line.color = "red"
        trace.line.dash = "dash"
        trace.line.width = 5
    elif trace.name == str(prev_season):
        trace.line.color = "black"
        trace.line.dash = "solid"
        trace.line.width = 3

# Layout and hover
fig_ratio_w_c.update_traces(
    hovertemplate="%{x|%d-%b}<br>%{y:.2f}<extra>%{fullData.name}</extra>"
)
fig_ratio_w_c.update_layout(
    xaxis=dict(
        tickformat="%b",
        dtick="M1",
        hoverformat="%d-%b"
    ),
    yaxis_title="Price",
    template="plotly_white",
    height=700,
    width=1800,
    hovermode="x unified"
)

fig_ratio_w_c.show()

fig_ratio_w_c_html = fig_ratio_w_c.to_html(include_plotlyjs="cdn", full_html=True)


In [0]:
seasons = ratio_w_c["season"].unique()
line_styles = {
    "ratio": dict(dash="solid", width=2),
    "WHEAT": dict(dash="dot", width=2),
    "CORN": dict(dash="dash", width=2)
}
color_map = {season: px.colors.qualitative.Plotly[i % len(px.colors.qualitative.Plotly)]
             for i, season in enumerate(seasons)}

fig_wheat_corn = go.Figure()

# Add all traces
for season in seasons:
    df_season = ratio_w_c[ratio_w_c["season"] == season]
    # ratio
    fig_wheat_corn.add_trace(go.Scatter(
        x=df_season["virtual_date"],
        y=df_season["ratio"],
        mode="lines",
        name=f"{season} - Ratio",
        line=dict(color=color_map[season], **line_styles["ratio"]),
        yaxis="y1",
        visible=True
    ))
    # WHEAT
    fig_wheat_corn.add_trace(go.Scatter(
        x=df_season["virtual_date"],
        y=df_season["WHEAT"],
        mode="lines",
        name=f"{season} - WHEAT",
        line=dict(color=color_map[season], **line_styles["WHEAT"]),
        yaxis="y2",
        visible=True
    ))
    # CORN
    fig_wheat_corn.add_trace(go.Scatter(
        x=df_season["virtual_date"],
        y=df_season["CORN"],
        mode="lines",
        name=f"{season} - CORN",
        line=dict(color=color_map[season], **line_styles["CORN"]),
        yaxis="y2",
        visible=True
    ))

# Create buttons to toggle each season
buttons = []
for i, season in enumerate(seasons):
    # Determine which traces belong to this season
    visible = [False] * len(fig_wheat_corn.data)
    for j, trace in enumerate(fig_wheat_corn.data):
        if trace.name.startswith(str(season)):
            visible[j] = True
    buttons.append(dict(
        label=str(season),
        method="update",
        args=[{"visible": visible},
              {"title": f"MATBA WHEAT DEC/ CORN DEC (Showing {season})"}]
    ))
buttons.append(dict(
    label="Show All",
    method="update",
    args=[{"visible": [True]*len(fig_wheat_corn.data)},  # all traces visible
          {"title": "WHEAT DEC / CORN DEC (All Seasons)"}]
))

# Layout
fig_wheat_corn.update_layout(
    updatemenus=[dict(
        type="dropdown",
        active=0,
        buttons=buttons,
        x=0.95,
        y=1.15,
        xanchor="left",
        yanchor="top"
    )],
    title=f"WHEAT DEC / CORN DEC",
    template="plotly_white",
    height=700,
    width=1800,
    hovermode="x unified",
    xaxis=dict(tickformat="%b", dtick="M1", hoverformat="%d-%b"),
    yaxis=dict(title="Ratio", side="left"),
    yaxis2=dict(title="Prices (WHEAT & CORN)", overlaying="y", side="right")
)

fig_wheat_corn.show()

fig_wheat_corn_html = fig_wheat_corn.to_html(include_plotlyjs="cdn", full_html=True)




## DEC WHEAT / NOV SB

In [0]:
dec_w = wheat_matba[wheat_matba["contract_month"] == 12].copy()
dec_sb= soy_matba[soy_matba["contract_month"] == 11].copy()

dec_w_match = (dec_w.groupby("date")
                .apply(lambda x: x.loc[x["contract_date"] >= x.name].sort_values("contract_date").head(1))
                .reset_index(drop=True))

dec_sb_match = (dec_sb.groupby("date")
                .apply(lambda x: x.loc[x["contract_date"] >= x.name].sort_values("contract_date").head(1))
                .reset_index(drop=True))

dec_sb_match.drop(["contract_year", "contract_month", "contract_date","virtual_date"], axis=1, inplace=True)
dec_sb_match.rename(columns={"value": "SB"}, inplace=True)


dec_w_match.drop(["contract_year", "contract_month", "contract_date","virtual_date"], axis=1, inplace=True)
dec_w_match.rename(columns={"value": "WHEAT"}, inplace=True)


# Merge dec_w + Jul
ratio_w_sb = pd.merge(dec_sb_match,dec_w_match, on="date")

ratio_w_sb = ratio_w_sb[ratio_w_sb['SB'] != 0]
ratio_w_sb = ratio_w_sb[ratio_w_sb['WHEAT'] != 0]

# Compute spread
ratio_w_sb["ratio"] = ratio_w_sb["WHEAT"]/ratio_w_sb["SB"]
ratio_w_sb['day']=ratio_w_sb['date'].dt.day
ratio_w_sb['month']=ratio_w_sb['date'].dt.month
ratio_w_sb['year']=ratio_w_sb['date'].dt.year
ratio_w_sb['virtual_date'] = pd.to_datetime(
    {'year': 2000, 'month': ratio_w_sb['month'], 'day': ratio_w_sb['day']},
    errors='coerce'  # This will convert invalid dates (e.g. Feb 30) to NaT
)

ratio_w_sb = ratio_w_sb[(ratio_w_sb['month'] <= 11)]

ratio_w_sb['season'] = ratio_w_sb['year']

# Drop rows with invalid virtual dates
ratio_w_sb = ratio_w_sb.dropna(subset=['virtual_date'])
ratio_w_sb

In [0]:
# Create the base plot
fig_ratio_w_sb = px.line(
    ratio_w_sb,
    x="virtual_date",
    y="ratio",
    color="season",
    labels={
        "virtual_date": "",
        "ratio": "MATBA Ratios DEC WHEAT/DEC SB ",
        "season": "season"
    },
    title="DEC WHEAT/DEC SB",
)

# Compute average across seasons
avg_df = ratio_w_sb.groupby("virtual_date", as_index=False)["ratio"].mean()
fig_ratio_w_sb.add_scatter(
    x=avg_df["virtual_date"],
    y=avg_df["ratio"],
    mode="lines",
    name="Average",
    line=dict(color="gray", width=3, dash="dash")  # dashed gray line
)

# Identify last two seasons
seasons_sorted = sorted(ratio_w_sb["season"].unique())
last_season = seasons_sorted[-1]
prev_season = seasons_sorted[-2]

# Update traces: highlight last and previous seasons
for trace in fig_ratio_w_sb.data:
    if trace.name == str(last_season):
        trace.line.color = "red"
        trace.line.dash = "dash"
        trace.line.width = 5
    elif trace.name == str(prev_season):
        trace.line.color = "black"
        trace.line.dash = "solid"
        trace.line.width = 3

# Layout and hover
fig_ratio_w_sb.update_traces(
    hovertemplate="%{x|%d-%b}<br>%{y:.2f}<extra>%{fullData.name}</extra>"
)
fig_ratio_w_sb.update_layout(
    xaxis=dict(
        tickformat="%b",
        dtick="M1",
        hoverformat="%d-%b"
    ),
    yaxis_title="Price",
    template="plotly_white",
    height=700,
    width=1800,
    hovermode="x unified"
)

fig_ratio_w_sb.show()

fig_ratio_w_sb_html = fig_ratio_w_sb.to_html(include_plotlyjs="cdn", full_html=True)


In [0]:
seasons = ratio_w_sb["season"].unique()
line_styles = {
    "ratio": dict(dash="solid", width=2),
    "WHEAT": dict(dash="dot", width=2),
    "SB": dict(dash="dash", width=2)
}
color_map = {season: px.colors.qualitative.Plotly[i % len(px.colors.qualitative.Plotly)]
             for i, season in enumerate(seasons)}

fig_wheat_sb = go.Figure()

# Add all traces
for season in seasons:
    df_season = ratio_w_sb[ratio_w_sb["season"] == season]
    # ratio
    fig_wheat_sb.add_trace(go.Scatter(
        x=df_season["virtual_date"],
        y=df_season["ratio"],
        mode="lines",
        name=f"{season} - Ratio",
        line=dict(color=color_map[season], **line_styles["ratio"]),
        yaxis="y1",
        visible=True
    ))
    # WHEAT
    fig_wheat_sb.add_trace(go.Scatter(
        x=df_season["virtual_date"],
        y=df_season["WHEAT"],
        mode="lines",
        name=f"{season} - WHEAT",
        line=dict(color=color_map[season], **line_styles["WHEAT"]),
        yaxis="y2",
        visible=True
    ))
    # CORN
    fig_wheat_sb.add_trace(go.Scatter(
        x=df_season["virtual_date"],
        y=df_season["SB"],
        mode="lines",
        name=f"{season} - SB",
        line=dict(color=color_map[season], **line_styles["SB"]),
        yaxis="y2",
        visible=True
    ))

# Create buttons to toggle each season
buttons = []
for i, season in enumerate(seasons):
    # Determine which traces belong to this season
    visible = [False] * len(fig_wheat_sb.data)
    for j, trace in enumerate(fig_wheat_sb.data):
        if trace.name.startswith(str(season)):
            visible[j] = True
    buttons.append(dict(
        label=str(season),
        method="update",
        args=[{"visible": visible},
              {"title": f"MATBA WHEAT DEC/ SB DEC (Showing {season})"}]
    ))
buttons.append(dict(
    label="Show All",
    method="update",
    args=[{"visible": [True]*len(fig_wheat_sb.data)},  # all traces visible
          {"title": "WHEAT DEC / SB DEC (All Seasons)"}]
))

# Layout
fig_wheat_sb.update_layout(
    updatemenus=[dict(
        type="dropdown",
        active=0,
        buttons=buttons,
        x=0.95,
        y=1.15,
        xanchor="left",
        yanchor="top"
    )],
    title=f"WHEAT DEC / SB DEC",
    template="plotly_white",
    height=700,
    width=1800,
    hovermode="x unified",
    xaxis=dict(tickformat="%b", dtick="M1", hoverformat="%d-%b"),
    yaxis=dict(title="Ratio", side="left"),
    yaxis2=dict(title="Prices (WHEAT & SB)", overlaying="y", side="right")
)

fig_wheat_sb.show()

fig_wheat_sb_html = fig_wheat_sb.to_html(include_plotlyjs="cdn", full_html=True)




In [0]:
ratio_html  = f"""
<!DOCTYPE html>
<html>
<head>
    <title>MATBA SB/CORN</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            padding: 20px;
        }}
        h2 {{
            margin-top: 40px;
            color: #2c3e50;
        }}
        .chart-container {{
            margin-bottom: 50px;
        }}
    </style>
</head>
<body>
    <h1>MATBA RATIO SB/CORN</h1>
    <h2>Ratio</h2>
    <div class="chart-container">{ratio_apr_july_html}</div>
    <h2>Ratio and Prices</h2>
    <div class="chart-container">{fig_html}</div>
    

    <h1>MATBA RATIO DEC WHEAT/ DEC CORN</h1>
    <h2>Ratio</h2>
    <div class="chart-container">{fig_ratio_w_c_html}</div>
    <h2>Ratio and Prices</h2>
    <div class="chart-container">{fig_wheat_corn_html}</div>

    <h1>MATBA RATIO DEC WHEAT/ DEC SB</h1>
    <h2>Ratio</h2>
    <div class="chart-container">{fig_ratio_w_sb_html}</div>
    <h2>Ratio and Prices</h2>
    <div class="chart-container">{fig_wheat_sb_html}</div>


</body>
</html>
"""
ratio_html_report_bytes = ratio_html.encode("utf-8")

grains=['florian.girardi-ext@ldc.com','Roman.Avramishin@LDC.com','juan.garciafuentes@ldc.com','gonzalo.lascombes@ldc.com','juan.carnemolla@LDC.com','valentin.chiesa@ldc.com','mateo.vergniaud@ldc.com']
test_2=['florian.girardi-ext@ldc.com','mateo.vergniaud@ldc.com']
      
# Send email with embedded chart and table
LDCDataAccessLayerPy.mail.mail_send(
    to=grains,
    subject=f'MATBA Ratios {datetime.today().strftime("%d-%m")}',
    from_addr="florian.girardi-ext@ldc.com",
    mime_type="html",
    body="Please find the attached report.", attachment={"matba_ratios.html": ratio_html_report_bytes}
)


